In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Student Performance Prediction System\n",
    "\n",
    "## Project Overview\n",
    "This notebook presents a comprehensive analysis of student performance prediction using various machine learning techniques. The goal is to predict final academic performance based on factors such as attendance, study hours, internal marks, and assignment scores."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Import Libraries and Load Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import necessary libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import StandardScaler, LabelEncoder\n",
    "from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor\n",
    "from sklearn.linear_model import LinearRegression, Ridge, Lasso\n",
    "from sklearn.svm import SVR\n",
    "from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n",
    "from sklearn.model_selection import cross_val_score\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style for visualizations\n",
    "plt.style.use('seaborn-v0_8')\n",
    "sns.set_palette(\"husl\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load or generate dataset\n",
    "try:\n",
    "    df = pd.read_csv('data/student_performance.csv')\n",
    "    print(\"Dataset loaded successfully!\")\n",
    "except FileNotFoundError:\n",
    "    print(\"Generating sample dataset...\")\n",
    "    \n",
    "    # Generate synthetic data\n",
    "    np.random.seed(42)\n",
    "    n_samples = 1000\n",
    "    \n",
    "    data = {\n",
    "        'student_id': range(1, n_samples + 1),\n",
    "        'gender': np.random.choice(['Male', 'Female'], n_samples),\n",
    "        'age': np.random.randint(18, 25, n_samples),\n",
    "        'study_hours_per_week': np.random.uniform(5, 40, n_samples),\n",
    "        'attendance_percentage': np.random.uniform(60, 100, n_samples),\n",
    "        'assignment_score': np.random.uniform(50, 100, n_samples),\n",
    "        'internal_marks': np.random.uniform(40, 95, n_samples),\n",
    "        'previous_gpa': np.random.uniform(2.0, 4.0, n_samples),\n",
    "        'extracurricular_activities': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),\n",
    "        'parental_education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples)\n",
    "    }\n",
    "    \n",
    "    df = pd.DataFrame(data)\n",
    "    \n",
    "    # Calculate final score\n",
    "    df['final_score'] = (\n",
    "        0.3 * df['study_hours_per_week'] / 40 * 100 +\n",
    "        0.25 * df['attendance_percentage'] +\n",
    "        0.2 * df['assignment_score'] +\n",
    "        0.15 * df['internal_marks'] +\n",
    "        0.1 * df['previous_gpa'] / 4 * 100\n",
    "    ) + np.random.normal(0, 5, n_samples)\n",
    "    \n",
    "    df['final_score'] = df['final_score'].clip(0, 100)\n",
    "    \n",
    "    # Categorize performance\n",
    "    df['performance_category'] = pd.cut(\n",
    "        df['final_score'],\n",
    "        bins=[0, 60, 75, 90, 100],\n",
    "        labels=['Poor', 'Average', 'Good', 'Excellent']\n",
    "    )\n",
    "    \n",
    "    # Save dataset\n",
    "    df.to_csv('data/student_performance.csv', index=False)\n",
    "    print(\"Sample dataset generated and saved!\")\n",
    "\n",
    "print(f\"Dataset shape: {df.shape}\")\n",
    "print(\"\\nFirst 5 rows:\")\n",
    "display(df.head())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Exploratory Data Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic statistics\n",
    "print(\"Basic Statistics:\")\n",
    "display(df.describe())\n",
    "\n",
    "# Data types and missing values\n",
    "print(\"\\nData Types:\")\n",
    "print(df.dtypes)\n",
    "\n",
    "print(\"\\nMissing Values:\")\n",
    "print(df.isnull().sum())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Distribution of final scores\n",
    "plt.figure(figsize=(12, 4))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "sns.histplot(df['final_score'], bins=30, kde=True)\n",
    "plt.title('Distribution of Final Scores')\n",
    "plt.xlabel('Final Score')\n",
    "plt.ylabel('Frequency')\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "df['performance_category'].value_counts().plot(kind='pie', autopct='%1.1f%%')\n",
    "plt.title('Performance Category Distribution')\n",
    "plt.ylabel('')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "numeric_cols = df.select_dtypes(include=[np.number]).columns\n",
    "correlation_matrix = df[numeric_cols].corr()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', center=0)\n",
    "plt.title('Correlation Matrix')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Relationships between features and target\n",
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "\n",
    "# Study hours vs final score\n",
    "sns.scatterplot(data=df, x='study_hours_per_week', y='final_score', hue='gender', ax=axes[0,0])\n",
    "axes[0,0].set_title('Study Hours vs Final Score')\n",
    "\n",
    "# Attendance vs final score\n",
    "sns.scatterplot(data=df, x='attendance_percentage', y='final_score', ax=axes[0,1])\n",
    "axes[0,1].set_title('Attendance vs Final Score')\n",
    "\n",
    "# Assignment score vs final score\n",
    "sns.scatterplot(data=df, x='assignment_score', y='final_score', ax=axes[1,0])\n",
    "axes[1,0].set_title('Assignment Score vs Final Score')\n",
    "\n",
    "# Box plot of scores by performance category\n",
    "sns.boxplot(data=df, x='performance_category', y='final_score', ax=axes[1,1])\n",
    "axes[1,1].set_title('Final Score by Performance Category')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Data Preprocessing"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Encode categorical variables\n",
    "le_gender = LabelEncoder()\n",
    "le_parental_edu = LabelEncoder()\n",
    "le_performance = LabelEncoder()\n",
    "\n",
    "df['gender_encoded'] = le_gender.fit_transform(df['gender'])\n",
    "df['parental_education_encoded'] = le_parental_edu.fit_transform(df['parental_education'])\n",
    "df['performance_encoded'] = le_performance.fit_transform(df['performance_category'])\n",
    "\n",
    "# Select features for modeling\n",
    "feature_columns = [\n",
    "    'study_hours_per_week', 'attendance_percentage', 'assignment_score',\n",
    "    'internal_marks', 'previous_gpa', 'extracurricular_activities',\n",
    "    'gender_encoded', 'parental_education_encoded'\n",
    "]\n",
    "\n",
    "X = df[feature_columns]\n",
    "y = df['final_score']\n",
    "\n",
    "print(f\"Feature matrix shape: {X.shape}\")\n",
    "print(f\"Target vector shape: {y.shape}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split the data\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42\n",
    ")\n",
    "\n",
    "# Scale features\n",
    "scaler = StandardScaler()\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_test_scaled = scaler.transform(X_test)\n",
    "\n",
    "print(f\"Training set size: {X_train.shape[0]}\")\n",
    "print(f\"Test set size: {X_test.shape[0]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Model Training and Evaluation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize models\n",
    "models = {\n",
    "    'Linear Regression': LinearRegression(),\n",
    "    'Ridge Regression': Ridge(alpha=1.0),\n",
    "    'Lasso Regression': Lasso(alpha=0.1),\n",
    "    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),\n",
    "    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),\n",
    "    'SVR': SVR(kernel='rbf')\n",
    "}\n",
    "\n",
    "# Train and evaluate models\n",
    "results = {}\n",
    "\n",
    "for name, model in models.items():\n",
    "    print(f\"\\nTraining {name}...\")\n",
    "    \n",
    "    # Train the model\n",
    "    model.fit(X_train_scaled, y_train)\n",
    "    \n",
    "    # Make predictions\n",
    "    y_pred = model.predict(X_test_scaled)\n",
    "    \n",
    "    # Calculate metrics\n",
    "    mse = mean_squared_error(y_test, y_pred)\n",
    "    rmse = np.sqrt(mse)\n",
    "    mae = mean_absolute_error(y_test, y_pred)\n",
    "    r2 = r2_score(y_test, y_pred)\n",
    "    \n",
    "    # Cross-validation\n",
    "    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')\n",
    "    \n",
    "    # Store results\n",
    "    results[name] = {\n",
    "        'MSE': mse,\n",
    "        'RMSE': rmse,\n",
    "        'MAE': mae,\n",
    "        'R2': r2,\n",
    "        'CV_R2_Mean': cv_scores.mean(),\n",
    "        'CV_R2_Std': cv_scores.std()\n",
    "    }\n",
    "    \n",
    "    print(f\"R2 Score: {r2:.4f}\")\n",
    "    print(f\"RMSE: {rmse:.4f}\")\n",
    "    print(f\"MAE: {mae:.4f}\")\n",
    "    print(f\"CV R2 Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create comparison DataFrame\n",
    "results_df = pd.DataFrame(results).T\n",
    "print(\"Model Performance Comparison:\")\n",
    "display(results_df)\n",
    "\n",
    "# Find best model\n",
    "best_model_name = results_df['R2'].idxmax()\n",
    "best_r2 = results_df.loc[best_model_name, 'R2']\n",
    "print(f\"\\nBest Model: {best_model_name}\")\n",
    "print(f\"Best R2 Score: {best_r2:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize model comparison\n",
    "fig, axes = plt.subplots(1, 3, figsize=(18, 6))\n",
    "\n",
    "metrics = ['R2', 'RMSE', 'MAE']\n",
    "\n",
    "for idx, metric in enumerate(metrics):\n",
    "    values = results_df[metric]\n",
    "    colors = ['green' if name == best_model_name else 'lightblue' for name in results_df.index]\n",
    "    \n",
    "    bars = axes[idx].bar(range(len(results_df)), values, color=colors)\n",
    "    axes[idx].set_title(f'{metric} Comparison')\n",
    "    axes[idx].set_ylabel(metric)\n",
    "    axes[idx].set_xticks(range(len(results_df)))\n",
    "    axes[idx].set_xticklabels(results_df.index, rotation=45, ha='right')\n",
    "    \n",
    "    # Add value labels on bars\n",
    "    for bar, value in zip(bars, values):\n",
    "        height = bar.get_height()\n",
    "        axes[idx].text(bar.get_x() + bar.get_width()/2., height,\n",
    "                      f'{value:.3f}', ha='center', va='bottom')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Feature Importance Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get feature importance from the best model (if it's a tree-based model)\n",
    "best_model = models[best_model_name]\n",
    "\n",
    "if hasattr(best_model, 'feature_importances_'):\n",
    "    # Create feature importance DataFrame\n",
    "    importance_df = pd.DataFrame({\n",
    "        'Feature': feature_columns,\n",
    "        'Importance': best_model.feature_importances_\n",
    "    }).sort_values('Importance', ascending=False)\n",
    "    \n",
    "    # Plot feature importance\n",
    "    plt.figure(figsize=(10, 6))\n",
    "    sns.barplot(data=importance_df, x='Importance', y='Feature')\n",
    "    plt.title(f'Feature Importance - {best_model_name}')\n",
    "    plt.xlabel('Importance Score')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    \n",
    "    print(\"Feature Importance:\")\n",
    "    display(importance_df)\n",
    "else:\n",
    "    print(f\"{best_model_name} doesn't provide feature importance\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Model Evaluation on Test Set"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate best model on test set\n",
    "y_pred_best = best_model.predict(X_test_scaled)\n",
    "\n",
    "# Create residual plot\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 6))\n",
    "\n",
    "# Residual plot\n",
    "residuals = y_test - y_pred_best\n",
    "axes[0].scatter(y_pred_best, residuals, alpha=0.5)\n",
    "axes[0].axhline(y=0, color='r', linestyle='--')\n",
    "axes[0].set_xlabel('Predicted Values')\n",
    "axes[0].set_ylabel('Residuals')\n",
    "axes[0].set_title('Residual Plot')\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Prediction vs Actual\n",
    "axes[1].scatter(y_test, y_pred_best, alpha=0.5)\n",
    "axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)\n",
    "axes[1].set_xlabel('Actual Values')\n",
    "axes[1].set_ylabel('Predicted Values')\n",
    "axes[1].set_title('Actual vs Predicted')\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# Print final metrics\n",
    "print(f\"\\nFinal Model Evaluation ({best_model_name}):\")\n",
    "print(f\"R2 Score: {r2_score(y_test, y_pred_best):.4f}\")\n",
    "print(f\"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}\")\n",
    "print(f\"MAE: {mean_absolute_error(y_test, y_pred_best):.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Save Model and Scaler"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import joblib\n",
    "\n",
    "# Save the best model\n",
    "joblib.dump(best_model, 'best_model.pkl')\n",
    "print(\"Best model saved as 'best_model.pkl'\")\n",
    "\n",
    "# Save the scaler\n",
    "joblib.dump(scaler, 'scaler.pkl')\n",
    "print(\"Scaler saved as 'scaler.pkl'\")\n",
    "\n",
    "# Save feature columns for reference\n",
    "with open('feature_columns.txt', 'w') as f:\n",
    "    for col in feature_columns:\n",
    "        f.write(f\"{col}\\n\")\n",
    "print(\"Feature columns saved as 'feature_columns.txt'\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Conclusions and Insights\n",
    "\n",
    "### Key Findings:\n",
    "1. **Model Performance**: The {best_model_name} achieved the best performance with an R2 score of {best_r2:.4f}\n",
    "2. **Important Features**: Study hours, attendance, and assignment scores are the most influential factors\n",
    "3. **Predictive Power**: The model can effectively predict student performance with reasonable accuracy\n",
    "\n",
    "### Recommendations:\n",
    "1. Students should focus on maintaining consistent study hours and attendance\n",
    "2. Assignment completion significantly impacts final performance\n",
    "3. The model can be used to identify at-risk students early\n",
    "\n",
    "### Future Work:\n",
    "1. Collect more diverse data from different institutions\n",
    "2. Include additional features like socioeconomic factors\n",
    "3. Implement real-time monitoring system\n",
    "4. Develop mobile application for easy access"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

NameError: name 'null' is not defined